# 🧠 Modèle de Cohérence Multimodale (Image + Texte) — FROM SCRATCH

**But :** Prédire si l'image et le texte d'un meme parlent de la même chose  
**Dataset :** Facebook Hateful Memes (Kaggle)  
**Contrainte :** Aucun modèle pré-entraîné — tout est codé from scratch  

**Architecture :**
- **Encodeur Image** : CNN 4 couches (Conv → BatchNorm → ReLU → MaxPool)
- **Encodeur Texte** : Embedding entraînable + LSTM Bidirectionnel
- **Fusion** : Concaténation + Produit élément par élément
- **Classifieur** : MLP → Sigmoid

> ⚙️ **Settings → Accelerator → GPU P100** avant de lancer !


## 📦 Partie 0 : Imports

In [1]:
import os
import json
import random
import numpy as np
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
import matplotlib.pyplot as plt

# Reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}")


✅ Device : cpu


## ⚙️ Partie 1 : Hyperparamètres

In [2]:
# --- Hyperparamètres ---
IMG_SIZE = 64          # Taille des images (64x64)
MAX_VOCAB = 5000       # Taille du vocabulaire
MAX_SEQ_LEN = 32       # Longueur max des séquences texte
EMBED_DIM = 128        # Dimension des embeddings
BATCH_SIZE = 32
NUM_EPOCHS = 15
LEARNING_RATE = 1e-3
TRAIN_RATIO = 0.8      # 80% train, 20% val

# Chemin du dataset sur Kaggle
DATASET_DIR = "/kaggle/input/facebook-hateful-meme-dataset"

print("✅ Hyperparamètres définis")


✅ Hyperparamètres définis


## 📂 Partie 2 : Chargement & Préparation du Dataset

On charge le dataset Facebook Hateful Memes, puis on crée :
- **Paires positives (label=1)** : image + son vrai texte → cohérent
- **Paires négatives (label=0)** : image + texte d'un autre meme → non cohérent


In [3]:
def find_jsonl_file(base_dir):
    """Cherche automatiquement le fichier jsonl dans le dataset."""
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith(".jsonl") and "train" in f.lower():
                return os.path.join(root, f)
            if f.endswith(".jsonl"):
                return os.path.join(root, f)
    for root, dirs, files in os.walk(base_dir):
        for f in files:
            if f.endswith(".json") and "train" in f.lower():
                return os.path.join(root, f)
    return None


def load_data(dataset_dir):
    """Charge les données depuis le dataset Kaggle."""
    jsonl_path = find_jsonl_file(dataset_dir)
    
    if jsonl_path is None:
        print("⚠️ Fichiers trouvés dans le dataset :")
        for root, dirs, files in os.walk(dataset_dir):
            for f in files[:20]:
                print(f"  {os.path.join(root, f)}")
        raise FileNotFoundError(f"Aucun fichier jsonl/json trouvé dans {dataset_dir}")
    
    print(f"📄 Fichier trouvé : {jsonl_path}")
    
    data = []
    with open(jsonl_path, "r") as f:
        for line in f:
            item = json.loads(line.strip())
            img_path = item.get("img", "")
            if not os.path.isabs(img_path):
                img_path = os.path.join(dataset_dir, img_path)
            
            if os.path.exists(img_path):
                data.append({
                    "img_path": img_path,
                    "text": item.get("text", ""),
                    "label": item.get("label", 0)
                })
    
    print(f"✅ {len(data)} exemples chargés")
    return data


def create_coherence_pairs(data, neg_ratio=1.0):
    """
    Crée le dataset de cohérence :
    - Paires positives (label=1) : image + son vrai texte
    - Paires négatives (label=0) : image + texte d'un autre meme
    """
    pairs = []
    
    # Paires positives
    for item in data:
        pairs.append({
            "img_path": item["img_path"],
            "text": item["text"],
            "label": 1
        })
    
    # Paires négatives
    n_neg = int(len(data) * neg_ratio)
    all_texts = [item["text"] for item in data]
    
    for i in range(n_neg):
        idx = i % len(data)
        neg_idx = random.randint(0, len(data) - 1)
        while neg_idx == idx:
            neg_idx = random.randint(0, len(data) - 1)
        
        pairs.append({
            "img_path": data[idx]["img_path"],
            "text": all_texts[neg_idx],
            "label": 0
        })
    
    random.shuffle(pairs)
    n_pos = sum(p["label"] for p in pairs)
    print(f"✅ Total paires : {len(pairs)} (positives: {n_pos}, négatives: {len(pairs) - n_pos})")
    return pairs


# --- Exécution ---
print("Chargement des données...")
raw_data = load_data(DATASET_DIR)
pairs = create_coherence_pairs(raw_data, neg_ratio=1.0)


Chargement des données...
⚠️ Fichiers trouvés dans le dataset :


FileNotFoundError: Aucun fichier jsonl/json trouvé dans /kaggle/input/facebook-hateful-meme-dataset

## 🔍 Partie 3 : Exploration rapide du dataset

In [ ]:
# Afficher quelques exemples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Exemples du dataset", fontsize=14, fontweight="bold")

for i, ax in enumerate(axes.flat):
    pair = pairs[i]
    img = Image.open(pair["img_path"]).convert("RGB")
    ax.imshow(img)
    label_str = "✅ Cohérent" if pair["label"] == 1 else "❌ Non cohérent"
    text_short = pair["text"][:40] + "..." if len(pair["text"]) > 40 else pair["text"]
    ax.set_title(f"{label_str}\n{text_short}", fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()


## ✂️ Partie 4 : Tokenizer FROM SCRATCH

On construit notre propre tokenizer :
- Lowercase + split par espaces/ponctuation
- Vocabulaire construit à partir du corpus
- Tokens spéciaux : `<PAD>`, `<UNK>`


In [ ]:
class SimpleTokenizer:
    """Tokenizer from scratch."""
    
    def __init__(self, max_vocab_size=5000, max_seq_len=32):
        self.max_vocab_size = max_vocab_size
        self.max_seq_len = max_seq_len
        self.word2idx = {"<PAD>": 0, "<UNK>": 1}
        self.idx2word = {0: "<PAD>", 1: "<UNK>"}
        self.vocab_size = 2
    
    def _tokenize(self, text):
        text = text.lower().strip()
        for char in ".,!?;:'\"()-/\\@#$%^&*[]{}|<>~`":
            text = text.replace(char, " ")
        tokens = text.split()
        return [t for t in tokens if len(t) > 0]
    
    def build_vocab(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(self._tokenize(text))
        
        most_common = counter.most_common(self.max_vocab_size - 2)
        for word, count in most_common:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word
        
        self.vocab_size = len(self.word2idx)
        print(f"✅ Vocabulaire : {self.vocab_size} mots")
    
    def encode(self, text):
        tokens = self._tokenize(text)
        indices = [self.word2idx.get(t, 1) for t in tokens]
        if len(indices) >= self.max_seq_len:
            indices = indices[:self.max_seq_len]
        else:
            indices = indices + [0] * (self.max_seq_len - len(indices))
        return indices


# --- Construire le tokenizer ---
tokenizer = SimpleTokenizer(max_vocab_size=MAX_VOCAB, max_seq_len=MAX_SEQ_LEN)
tokenizer.build_vocab([p["text"] for p in pairs])

# Test
example_text = pairs[0]["text"]
print(f"\nExemple : \"{example_text}\"")
print(f"Encodé  : {tokenizer.encode(example_text)}")


## 📊 Partie 5 : Dataset & DataLoaders PyTorch

In [ ]:
class MemeCoherenceDataset(Dataset):
    """Dataset PyTorch pour les paires image-texte."""
    
    def __init__(self, pairs, tokenizer, img_size=64):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.img_size = img_size
    
    def __len__(self):
        return len(self.pairs)
    
    def _load_image(self, path):
        try:
            img = Image.open(path).convert("RGB")
            img = img.resize((self.img_size, self.img_size))
            img = np.array(img, dtype=np.float32) / 255.0
            img = img.transpose(2, 0, 1)  # HWC → CHW
            return torch.tensor(img)
        except Exception:
            return torch.zeros(3, self.img_size, self.img_size)
    
    def __getitem__(self, idx):
        pair = self.pairs[idx]
        image = self._load_image(pair["img_path"])
        text_ids = torch.tensor(self.tokenizer.encode(pair["text"]), dtype=torch.long)
        label = torch.tensor(pair["label"], dtype=torch.float32)
        return image, text_ids, label


# --- Split train/val ---
split_idx = int(len(pairs) * TRAIN_RATIO)
train_pairs = pairs[:split_idx]
val_pairs = pairs[split_idx:]

train_dataset = MemeCoherenceDataset(train_pairs, tokenizer, img_size=IMG_SIZE)
val_dataset = MemeCoherenceDataset(val_pairs, tokenizer, img_size=IMG_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"✅ Train: {len(train_pairs)} | Val: {len(val_pairs)}")
print(f"   Batches — Train: {len(train_loader)} | Val: {len(val_loader)}")


## 🖼️ Partie 6 : Encodeur Image — CNN FROM SCRATCH

4 blocs convolutifs : `Conv2D → BatchNorm → ReLU → MaxPool`  
Puis Global Average Pooling → vecteur de dimension `embed_dim`


In [ ]:
class ImageEncoderCNN(nn.Module):
    """CNN from scratch pour encoder les images."""
    
    def __init__(self, embed_dim=128):
        super().__init__()
        
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(256, embed_dim)
    
    def forward(self, x):
        # x : (batch, 3, 64, 64)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))   # → (batch, 32, 32, 32)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))   # → (batch, 64, 16, 16)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))   # → (batch, 128, 8, 8)
        x = self.pool(F.relu(self.bn4(self.conv4(x))))   # → (batch, 256, 4, 4)
        
        x = x.mean(dim=[2, 3])  # Global Average Pooling → (batch, 256)
        x = self.dropout(x)
        x = self.fc(x)          # → (batch, embed_dim)
        return x

print("✅ ImageEncoderCNN défini")


## 📝 Partie 7 : Encodeur Texte — Embedding + LSTM FROM SCRATCH

- Embedding entraînable (PAS de Word2Vec / GloVe)
- LSTM Bidirectionnel
- Dernier hidden state comme représentation du texte


In [ ]:
class TextEncoderLSTM(nn.Module):
    """Encodeur texte from scratch : Embedding + BiLSTM."""
    
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_layers=1):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        
        self.fc = nn.Linear(hidden_dim * 2, embed_dim)
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # x : (batch, seq_len)
        embedded = self.dropout(self.embedding(x))       # → (batch, seq_len, embed_dim)
        output, (hidden, cell) = self.lstm(embedded)
        
        # Concaténer forward et backward
        hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)  # → (batch, hidden_dim*2)
        out = self.fc(hidden_cat)  # → (batch, embed_dim)
        return out

print("✅ TextEncoderLSTM défini")


## 🔗 Partie 8 : Modèle de Fusion Complet

Fusion des deux encodeurs :
- **Concaténation** : `[img_vec ; txt_vec]`
- **Produit élément par élément** : `img_vec * txt_vec`
- MLP → Sigmoid → probabilité de cohérence


In [ ]:
class MemeCoherenceModel(nn.Module):
    """Modèle complet de cohérence multimodale."""
    
    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        
        self.image_encoder = ImageEncoderCNN(embed_dim=embed_dim)
        self.text_encoder = TextEncoderLSTM(vocab_size, embed_dim=embed_dim)
        
        # Fusion : concat (embed_dim*2) + produit (embed_dim) = embed_dim*3
        fusion_dim = embed_dim * 3
        
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1)
        )
    
    def forward(self, images, text_ids):
        img_features = self.image_encoder(images)
        txt_features = self.text_encoder(text_ids)
        
        # Normalisation L2
        img_features = F.normalize(img_features, p=2, dim=1)
        txt_features = F.normalize(txt_features, p=2, dim=1)
        
        # Fusion
        concat = torch.cat([img_features, txt_features], dim=1)
        elementwise = img_features * txt_features
        fused = torch.cat([concat, elementwise], dim=1)
        
        logits = self.classifier(fused)
        return logits.squeeze(1)


# --- Instancier le modèle ---
model = MemeCoherenceModel(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Modèle créé — {total_params:,} paramètres")
print(f"\n{model}")


## 🏋️ Partie 9 : Fonctions d'Entraînement & Évaluation

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    for batch_idx, (images, text_ids, labels) in enumerate(dataloader):
        images = images.to(DEVICE)
        text_ids = text_ids.to(DEVICE)
        labels = labels.to(DEVICE)
        
        optimizer.zero_grad()
        logits = model(images, text_ids)
        loss = criterion(logits, labels)
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
        preds = (torch.sigmoid(logits) > 0.5).float()
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        if (batch_idx + 1) % 50 == 0:
            print(f"  Batch {batch_idx+1}/{len(dataloader)} — "
                  f"Loss: {loss.item():.4f} — Acc: {correct/total:.4f}")
    
    return total_loss / len(dataloader), correct / total


def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, text_ids, labels in dataloader:
            images = images.to(DEVICE)
            text_ids = text_ids.to(DEVICE)
            labels = labels.to(DEVICE)
            
            logits = model(images, text_ids)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    tp = ((all_preds == 1) & (all_labels == 1)).sum()
    fp = ((all_preds == 1) & (all_labels == 0)).sum()
    fn = ((all_preds == 0) & (all_labels == 1)).sum()
    
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    
    return total_loss / len(dataloader), correct / total, f1, precision, recall

print("✅ Fonctions train/eval définies")


## 🚀 Partie 10 : Entraînement

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=2, verbose=True
)

best_val_f1 = 0
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "val_f1": []}

for epoch in range(NUM_EPOCHS):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    print(f"{'='*50}")
    
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(model, val_loader, criterion)
    
    scheduler.step(val_loss)
    
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)
    
    print(f"  Train — Loss: {train_loss:.4f} | Acc: {train_acc:.4f}")
    print(f"  Val   — Loss: {val_loss:.4f} | Acc: {val_acc:.4f}")
    print(f"  Val   — F1: {val_f1:.4f} | Prec: {val_prec:.4f} | Rec: {val_rec:.4f}")
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "best_model.pth")
        print(f"  ★ Meilleur modèle sauvegardé ! (F1: {val_f1:.4f})")

print(f"\n✅ Entraînement terminé — Meilleur F1: {best_val_f1:.4f}")


## 📈 Partie 11 : Courbes d'Entraînement

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(history["train_loss"], label="Train", marker="o")
axes[0].plot(history["val_loss"], label="Val", marker="s")
axes[0].set_title("Loss", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history["train_acc"], label="Train", marker="o")
axes[1].plot(history["val_acc"], label="Val", marker="s")
axes[1].set_title("Accuracy", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(history["val_f1"], label="Val F1", marker="s", color="green")
axes[2].set_title("F1-Score (Validation)", fontsize=13, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 🏆 Partie 12 : Évaluation Finale

In [ ]:
# Charger le meilleur modèle
model.load_state_dict(torch.load("best_model.pth"))
val_loss, val_acc, val_f1, val_prec, val_rec = evaluate(model, val_loader, criterion)

print("=" * 50)
print(" RÉSULTATS FINAUX")
print("=" * 50)
print(f"  Accuracy  : {val_acc:.4f}")
print(f"  Precision : {val_prec:.4f}")
print(f"  Recall    : {val_rec:.4f}")
print(f"  F1-Score  : {val_f1:.4f}")


## 🔍 Partie 13 : Matrice de Confusion

In [ ]:
# Collecter toutes les prédictions
model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for images, text_ids, labels in val_loader:
        images = images.to(DEVICE)
        text_ids = text_ids.to(DEVICE)
        
        logits = model(images, text_ids)
        preds = (torch.sigmoid(logits) > 0.5).float()
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# Matrice de confusion
tp = ((all_preds == 1) & (all_labels == 1)).sum()
tn = ((all_preds == 0) & (all_labels == 0)).sum()
fp = ((all_preds == 1) & (all_labels == 0)).sum()
fn = ((all_preds == 0) & (all_labels == 1)).sum()

cm = np.array([[tn, fp], [fn, tp]])

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Non Cohérent", "Cohérent"])
ax.set_yticklabels(["Non Cohérent", "Cohérent"])
ax.set_xlabel("Prédit", fontsize=12)
ax.set_ylabel("Réel", fontsize=12)
ax.set_title("Matrice de Confusion", fontsize=14, fontweight="bold")

for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=18,
                color="white" if cm[i, j] > cm.max()/2 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.show()


## 🧪 Partie 14 : Test sur des Exemples

In [ ]:
def predict_coherence(model, tokenizer, img_path, text, img_size=64):
    """Prédit la cohérence entre une image et un texte."""
    model.eval()
    
    img = Image.open(img_path).convert("RGB")
    img = img.resize((img_size, img_size))
    img = np.array(img, dtype=np.float32) / 255.0
    img = img.transpose(2, 0, 1)
    img_tensor = torch.tensor(img).unsqueeze(0).to(DEVICE)
    
    text_ids = torch.tensor(tokenizer.encode(text), dtype=torch.long).unsqueeze(0).to(DEVICE)
    
    with torch.no_grad():
        logit = model(img_tensor, text_ids)
        prob = torch.sigmoid(logit).item()
    
    return prob


# --- Tester sur quelques exemples ---
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

test_samples = val_pairs[:6]
for i, ax in enumerate(axes.flat):
    pair = test_samples[i]
    prob = predict_coherence(model, tokenizer, pair["img_path"], pair["text"])
    
    img = Image.open(pair["img_path"]).convert("RGB")
    ax.imshow(img)
    
    real = "✅ Cohérent" if pair["label"] == 1 else "❌ Non cohérent"
    pred = "✅ Cohérent" if prob > 0.5 else "❌ Non cohérent"
    text_short = pair["text"][:35] + "..." if len(pair["text"]) > 35 else pair["text"]
    
    ax.set_title(f"Réel: {real}\nPrédit: {pred} ({prob:.2f})\n\"{text_short}\"", fontsize=9)
    ax.axis("off")

plt.suptitle("Prédictions sur des exemples de validation", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\n🎉 Notebook terminé ! Le modèle est prêt.")
